### Observability Foundations
Here we will be tracing each of these functions as a span!

Langsmith is the observability platform we will be using

and we will be focusing on tracing

1. create a new project in langsmith
2. trace an existing app
3. install `uv add langsmith`
4. add .emv vars 

export LANGSMITH_TRACING=true
export LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com
export LANGSMITH_API_KEY=<your-api-key>
export LANGSMITH_PROJECT="AI-Bootcamp"

5. loaf the dotenv
6. `from langsmith import traceable`
7. add traceable decorator to everything we eant to track of with `@traceable`

this `@traceable` now becomes a span in this trace which we will track wehn we run the function. 
it tracks: time, runtime, inputs, outputs, child spans (nested functions), etc.

trace -> span -> subspan

now when we run the code we will see all of this metadata in langsmith

here we can see all of the inpouts outpus, time each function took i.e each span, subspan, total trace

everything in langsmith is a run 
each run has a type
multiple types 
i.e embedding, retriever, prompt, llm


Thre `@traceable` can also accept parameters to ennhance the metadata it caputures

```
@traceable(
    name="embed_query", 
    run_type="embedding"
)
```
this allows langsmith to also calculate things like cost. 

now we can see whats going on at a glance 

and calulate price when prividing the follwoinf
```
  metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
    }
```

to calculate the cost we need to track the number of tokens used
to do this we use getcurrentruntree (only for embeddings or generations)

we add this within the function. 

```
 current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }
```




In [31]:
import openai
from qdrant_client import QdrantClient

from dotenv import load_dotenv
import os
load_dotenv("../../.env")

from langsmith import traceable, get_current_run_tree


### Embedding Function

In [32]:
@traceable(
    name="embed_query", 
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
    }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }

    return response.data[0].embedding

### Retrieval Function

In [33]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [34]:
@traceable(
    name="retrieve_data", 
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [35]:
@traceable(
    name="format_retrieved_context", 
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

### create prompt template function

In [36]:
@traceable(
    name="build_prompt", 
    run_type="prompt"
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt


### Generate answer function

In [37]:
from urllib3 import response

@traceable(
    name="generate_answer", 
    run_type="llm",
     metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-nano"
    }
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [38]:
@traceable(
    name="rag_pipeline"
)
def rag_pipeline(question, topk_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer


In [39]:
print(rag_pipeline("Do you have a usb fan that will help me cool down in hot summers?"))

Yes. We have USB-powered desk/personal fans that can help keep you cool in hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z, rating 4.7)
- USB powered with a 3.3 ft USB cable
- Speed controller lets you adjust from off/low/medium/high
- Good for cooling electronics in airflow-restricted spaces (router, TV box, modem, etc.)

2) HZD Desk Fan Rechargeable, Mini Portable Fan (ID: B0BXC72RLD, rating 4.3)
- Compact mini desk fan (USB powered)
- 3 speed levels (low/medium/high)
- Quiet operation (noise less than 50 dB noted)
- Includes a 4.9 ft USB cable; compatible with USB power sources (laptops, mobile power, AC adapters, car chargers)
- Note: it says the USB model does not come with a battery

If you tell me whether you want it mainly for personal comfort at your desk or for cooling a device (like a router/TV box), I can point you to the best match.
